# Days 5–6: Retrieval package and first LangGraph research agent

**Day 5** turned the Day 3–4 experiment scripts into reusable packages:
- `retrieval/`: BM25, dense, hybrid (RRF) and cross-encoder reranking behind one call, `retrieve(question, paper_id)`
- `eval/`: metrics plus a script that reproduces the held-out result

**Day 6** built a LangGraph agent on top of `retrieve()`: **decompose → retrieve → cited answer**,
with an evaluation that checks every citation against QASPER's gold evidence.

Runs in Colab (GPU runtime) from the project folder in Google Drive.

## Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
%cd /content/drive/MyDrive/research-engine
!pip install -q -r requirements.txt

In [ ]:
# The Groq key lives in Colab Secrets (never in a cell). `!python` commands read it from the environment.
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY").strip()

## Day 5: reproduce the held-out retrieval result

Pipeline: BM25 top-50 + `bge-small-en-v1.5` top-50 within the paper → RRF (k=60) →
`bge-reranker-v2-m3` (fp16) re-scores the top 30 → abstract chunks removed.

Result on the QASPER **test** split (1,309 questions), identical to the Day 4 run:

| system | hit@1 | hit@5 | evidence R@5 | evidence R@10 | MRR@10 |
|---|---|---|---|---|---|
| BM25 | 0.254 | 0.735 | 0.612 | 0.849 | 0.455 |
| Dense | 0.409 | 0.824 | 0.699 | 0.890 | 0.581 |
| Hybrid RRF | 0.370 | 0.814 | 0.692 | 0.896 | 0.557 |
| + reranker | 0.390 | 0.869 | 0.749 | 0.932 | 0.588 |
| **final (+ abstracts removed)** | **0.522** | **0.906** | **0.795** | **0.941** | **0.677** |

In [ ]:
!python -m eval.run_retrieval_eval --split test

### The one-call retriever the agent uses

In [ ]:
import json
from retrieval import retrieve

q = next(json.loads(l) for l in open("data/processed/qasper_questions.jsonl") if '"split": "test"' in l)
print("Question:", q["question"])
print("Gold chunks:", q["gold_chunk_ids"])
for h in retrieve(q["question"], q["paper_id"], k=5):
    print(h["rank"], h["chunk_id"], h["section_type"], h["text"][:100])

## Day 6: LangGraph research agent

```
question -> decompose (LLM, JSON: up to 3 sub-questions)
         -> retrieve  (retrieve() for the question + each sub-question)
         -> answer    (LLM answers from the evidence only, citing [E#] after each claim)
```

LLM: Groq `openai/gpt-oss-120b` with low reasoning effort. Citation labels are mapped back to chunk IDs,
so each citation can be checked against gold evidence.

In [ ]:
!python -m agent "What datasets do they use?" --paper-id 1909.00694

### Evaluation on 20 dev questions

A first run found that gpt-oss writes citations as `【E2】` rather than `[E2]`, so 60% of answers looked uncited.
After normalising citation brackets in the parser and asking for ASCII `[E2]` in the prompt, uncited answers dropped to 0%.

| metric | value |
|---|---|
| citation precision (cited passages that are gold) | 0.667 |
| citation hit (≥1 gold passage cited) | 0.70 |
| evidence recall: agent's merged search | 0.883 |
| evidence recall: one plain search, same k | 0.983 |
| declined | 0.10 |
| tokens / question | ~2,200 |

**Takeaways:**
- Merging sub-question results pushed some of the original question's best passages out of the evidence.
  Decomposition hurt retrieval here, which is fixed on Day 7 by always keeping the original question's top results.
- Two declines were too cautious: a yes/no question, and a number that likely sits in a table.
- QASPER marks only 1–2 paragraphs as gold, so some "wrong" citations may still be valid support.

In [ ]:
!python -m eval.run_agent_eval --n 20

In [ ]:
# Read the answers next to their citations and the gold evidence
import json
for line in open("results/agent_eval_dev_n20.jsonl"):
    r = json.loads(line)
    print("Q:", r["question"])
    print("A:", r["answer"])
    print("cited:", r["cited_chunk_ids"], "| gold:", r["gold_chunk_ids"])
    print("-" * 80)